In [2]:
import pandas as pd
import pyarrow.parquet as pq


In [8]:
# Inspect parquet file metadata
parquet_file = pq.ParquetFile('sample_counts_matrix.parquet')
print(parquet_file.metadata)

  created_by: parquet-cpp-arrow version 24.0.0
  num_columns: 518
  num_rows: 60660
  num_row_groups: 1
  format_version: 2.6
  serialized_size: 263737


In [14]:
# Load input parquet file containing RNA-seq raw counts
df = pd.read_parquet('sample_counts_matrix.parquet')

# Check first 5 rows and columns
df.iloc[:5, :5]

,TCGA-44-7661,TCGA-55-6979,TCGA-55-8203,TCGA-NJ-A55R,TCGA-78-8640
gene_id,,,,,
ENSG00000000003.15,2138,3103,4144,3444,997
ENSG00000000005.6,4,0,0,3,0
ENSG00000000419.13,2179,1030,2439,2569,1227
ENSG00000000457.14,830,489,1644,1977,852
ENSG00000000460.17,610,462,670,545,527


In [37]:
# Set gene_id as index
if 'gene_id' in df.columns:
    df = df.set_index('gene_id')
    
# Inspect number of genes and samples
print("Dimensions (Genes, Samples):", df.shape)

# Inspect low-expression genes
genes_zero = (df == 0).all(axis=1).sum()
genes_low =(df < 10).all(axis=1).sum()

print(f"Genes with 0 raw counts in all samples: {genes_zero}")
print(f"Genes with <10 raw counts in all samples: {genes_low}")

Dimensions (Genes, Samples): (60660, 517)
Genes with 0 raw counts in all samples: 1379
Genes with <10 raw counts in all samples: 9716


In [40]:
# NA treatment

total_nas = df.isna().sum().sum()
print(f"Total de valores NA en toda la matriz: {total_nas}")

filas_con_na = df.isna().any(axis=1).sum()
print(f"Total de filas (genes) con al menos un NA: {filas_con_na}")

columnas_con_na = df.isna().any(axis=0).sum()
print(f"Total de columnas (muestras) con al menos un NA: {columnas_con_na}")

columna_con_na = df.columns[df.isna().any()].tolist()
print("La columna con NA es:", columna_con_na)




Total de valores NA en toda la matriz: 18197
Total de filas (genes) con al menos un NA: 18197
Total de columnas (muestras) con al menos un NA: 1
La columna con NA es: ['TCGA-55-8619']


In [44]:
primera_fila = df.iloc[0]

# Muestra los primeros 5 valores y los últimos 5 valores de esa fila
print("--- PRIMEROS 5 VALORES ---")
print(primera_fila.head())

print("\n--- ÚLTIMOS 5 VALORES ---")
print(primera_fila.tail())

posicion = df.columns.get_loc(col_na)

print(f"La columna '{col_na}' está en la posición número: {posicion}")

--- PRIMEROS 5 VALORES ---
TCGA-44-7661    2138.0
TCGA-55-6979    3103.0
TCGA-55-8203    4144.0
TCGA-NJ-A55R    3444.0
TCGA-78-8640     997.0
Name: ENSG00000000003.15, dtype: float64

--- ÚLTIMOS 5 VALORES ---
TCGA-86-8671    1811.0
TCGA-78-7536    4409.0
TCGA-MP-A4SW    3440.0
TCGA-44-7662    3513.0
TCGA-50-5049    1638.0
Name: ENSG00000000003.15, dtype: float64
La columna 'TCGA-55-8619' está en la posición número: 268


In [45]:
df.iloc[:5, posicion-2 : posicion+3]

,TCGA-50-5944,TCGA-67-3771,TCGA-55-8619,TCGA-55-A48Z,TCGA-53-A4EZ
gene_id,,,,,
ENSG00000000003.15,3266,1053,1020.0,6832,4627
ENSG00000000005.6,2,1,1.0,0,0
ENSG00000000419.13,1015,1817,831.0,1608,1318
ENSG00000000457.14,626,598,518.0,978,691
ENSG00000000460.17,147,339,132.0,409,354
